In [25]:
import math as mt
from collections import Counter

# =========================================================
# 1. OUTILS MATHÉMATIQUES (Mesures du désordre)
# =========================================================
  #Calcule le désordre global (pour ID3 et C4.5).
def calcule_entropie(y):
    nb_cas = len(y)
    counter = Counter(y) # Compte combien de OUI et de NON
    entropie = 0
    for i in counter.values():
        pi = i / nb_cas # Probabilité d'une classe
        entropie -= pi * mt.log2(pi) # Formule mathématique de Shannon
    return entropie

  #Calcule l'impureté (pour CART)
def calcule_gini(y):
    nb_cas = len(y)
    if nb_cas == 0: return 0
    counter = Counter(y)
    somme_carres = 0
    for i in counter.values():
        pi = i / nb_cas
        somme_carres += pi**2 # On somme les probabilités au carré
    return 1 - somme_carres # Gini = 1 - Somme des carrés

# =========================================================
# 2. LOGIQUE DES GAINS
# =========================================================
  # Mesure le gain d'information simple pour ID3
def calcule_gain_id3(X, y, idx):
    h_parent = calcule_entropie(y)
    colonne = [ligne[idx] for ligne in X] # On extrait la colonne testée
    v_unique = set(colonne) # On liste les catégories
    h_ponderee = 0
    for val in v_unique:
        # On crée un petit groupe pour chaque catégorie
        sous_y = [y[i] for i in range(len(y)) if X[i][idx] == val]
        # On fait la moyenne pondérée du désordre des enfants
        h_ponderee += (len(sous_y) / len(y)) * calcule_entropie(sous_y)
    return h_parent - h_ponderee # Gain = désordre avant - désordre après

def calcule_gain_ratio_c45(X, y, idx):
    h_parent = calcule_entropie(y)
    colonne = [ligne[idx] for ligne in X]
    v_unique = set(colonne)
    h_ponderee = 0
    split_info = 0 # Mesure à quel point la colonne divise les données
    for val in v_unique:
        sous_y = [y[i] for i in range(len(y)) if X[i][idx] == val]
        poids = len(sous_y) / len(y)
        h_ponderee += poids * calcule_entropie(sous_y)
        if poids > 0:
            split_info -= poids * mt.log2(poids) # Entropie de la colonne elle-même
    gain = h_parent - h_ponderee
    # Gain Ratio = Gain / Information de scission
    return gain / split_info if split_info != 0 else 0

def calcule_gain_gini_cart(X, y, idx):
    g_parent = calcule_gini(y)
    colonne = [ligne[idx] for ligne in X]
    v_unique = set(colonne)
    g_pondere = 0
    for val in v_unique:
        sous_y = [y[i] for i in range(len(y)) if X[i][idx] == val]
        # Moyenne pondérée du Gini des enfants
        g_pondere += (len(sous_y) / len(y)) * calcule_gini(sous_y)
    return g_parent - g_pondere

# =========================================================
# 3. FONCTION D'AFFICHAGE (Construction de l'arbre)
# =========================================================

def afficher_arbre(X, y, indices_dispos, noms_features, type_algo="ID3", profondeur=0):
    """ Construit l'arbre par récursion et l'affiche ligne par ligne. """
    espace = "    " * profondeur # Crée l'indentation visuelle

    #  Si tout le groupe a le même résultat (OUI ou NON)
    if len(set(y)) == 1 or not indices_dispos:
        resultat = Counter(y).most_common(1)[0][0] # On prend la majorité
        print(f"{espace}  => RÉPONSE : {resultat}")
        return

    # SÉLECTION : On cherche quel index de colonne donne le meilleur score
    meilleur_score = -1
    idx_gagnant = None

    for idx in indices_dispos:
        if type_algo == "ID3": score = calcule_gain_id3(X, y, idx)
        elif type_algo == "C4.5": score = calcule_gain_ratio_c45(X, y, idx)
        else: score = calcule_gain_gini_cart(X, y, idx)

        if score > meilleur_score:
            meilleur_score = score
            idx_gagnant = idx

    # On affiche la question choisie
    print(f"{espace}[ {type_algo} : {noms_features[idx_gagnant]} ? ]")

    valeurs_uniques = set([ligne[idx_gagnant] for ligne in X])
    # On prépare les indices restants (on retire la question qu'on vient de poser)
    nouveaux_indices = [i for i in indices_dispos if i != idx_gagnant]

    # Pour chaque réponse possible, on crée une nouvelle branche
    for val in valeurs_uniques:
        print(f"{espace}  |-- {val} :")
        # On filtre les données pour ne garder que ce qui correspond à la branche
        X_f = [X[i] for i in range(len(X)) if X[i][idx_gagnant] == val]
        y_f = [y[i] for i in range(len(y)) if X[i][idx_gagnant] == val]
        # On descend d'un niveau (récursion)
        afficher_arbre(X_f, y_f, nouveaux_indices, noms_features, type_algo, profondeur + 1)

# =========================================================
# 4. EXÉCUTION
# =========================================================

X_train = [
    ["Jeune", "Élevé", "Oui"], ["Jeune", "Élevé", "Non"], ["Moyen", "Élevé", "Non"],
    ["Âgé", "Faible", "Non"], ["Âgé", "Faible", "Oui"], ["Âgé", "Faible", "Oui"],
    ["Moyen", "Faible", "Oui"], ["Jeune", "Faible", "Non"]
]
y_train = ["OUI", "OUI", "OUI", "NON", "NON", "NON", "OUI", "NON"]
features = ["Âge", "Revenu", "Étudiant ?"]
indices = [0, 1, 2]

print("--- COMPARAISON DES ALGORITHMES ---\n")
afficher_arbre(X_train, y_train, indices, features, "ID3")
print("\n")
afficher_arbre(X_train, y_train, indices, features, "C4.5")
print("\n")
afficher_arbre(X_train, y_train, indices, features, "CART")

--- COMPARAISON DES ALGORITHMES ---

1. RÉSULTAT ID3 (Entropie)
[ ID3 : Âge ? ]
  |-- Moyen :
      => RÉPONSE : OUI
  |-- Âgé :
      => RÉPONSE : NON
  |-- Jeune :
    [ ID3 : Revenu ? ]
      |-- Faible :
          => RÉPONSE : NON
      |-- Élevé :
          => RÉPONSE : OUI

2. RÉSULTAT C4.5 (Gain Ratio)
[ C4.5 : Revenu ? ]
  |-- Faible :
    [ C4.5 : Âge ? ]
      |-- Âgé :
          => RÉPONSE : NON
      |-- Moyen :
          => RÉPONSE : OUI
      |-- Jeune :
          => RÉPONSE : NON
  |-- Élevé :
      => RÉPONSE : OUI

3. RÉSULTAT CART (Gini)
[ CART : Âge ? ]
  |-- Moyen :
      => RÉPONSE : OUI
  |-- Âgé :
      => RÉPONSE : NON
  |-- Jeune :
    [ CART : Revenu ? ]
      |-- Faible :
          => RÉPONSE : NON
      |-- Élevé :
          => RÉPONSE : OUI
